# 05 - Smoke test: every arm, one fold, a deliberately tiny budget

**What this step does.** Runs all four arms on a single fold at a budget far
too small to produce a real result, then measures how long a real run would
take. Nothing here is a finding.

**Why it exists.** The full matrix is **360 runs**. Discovering a shape
mismatch, a device error or a silently-broken arm on run 250 of 360 would cost
hours. A smoke test buys that insurance for a few minutes.

It answers four questions:

1. **Does every arm execute at all?** Arm A pretrains on three domains at once,
   arm D on one, arm C trains end to end - three different code paths.
2. **Are the numbers sane?** Not good, but plausible: above the no-model
   baseline, below the from-scratch ceiling, no NaNs.
3. **How long will the real thing take?** Measured, not guessed.
4. **Does the harness resume correctly?** A sweep that cannot restart after a
   crash is a sweep you cannot run overnight.

**The budget is not a result.** 30 pretraining steps instead of 300, 300 probe
epochs instead of 3000. At this budget the probe will not converge and the
encoder will barely be trained. Any ranking between arms here is noise, and
the notebook says so at every point where a reader might be tempted.

**What could go wrong.**
- **Reading smoke-test numbers as results.** The single biggest risk, so
  convergence flags are printed everywhere.
- **The target leaking into its own pretraining sources.** Asserted below.
- **A run harness that resumes by re-running.** Tested explicitly.

## Setup

Note `src/__init__.py` sets `CUBLAS_WORKSPACE_CONFIG` on import, and
`set_seed` enables deterministic CUDA kernels. Without both, GIN's scatter-add
aggregation gives different answers run to run -- the same config scored
0.7823, 0.7731 and 0.7694 on three consecutive runs before this was fixed.

In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

import os

import numpy as np
import pandas as pd
import torch

from src import arms, config, metrics, pipeline, plotting, splits
from src.config import RunConfig
from src.seeding import set_seed

SEED = 0
set_seed(SEED)
plotting.use_project_style()
device = config.get_device()
pd.set_option("display.width", 200)

print("device                  :", device)
print("CUBLAS_WORKSPACE_CONFIG :", os.environ.get("CUBLAS_WORKSPACE_CONFIG"))
print("deterministic algorithms:", torch.are_deterministic_algorithms_enabled())

device                  : cuda
CUBLAS_WORKSPACE_CONFIG : :4096:8
deterministic algorithms: True


## 1. The shape of the matrix

Leave-one-domain-out gives four folds, one per domain. Each fold runs at three
seeds and five label fractions, and each of those combinations runs:

- **arm A** - pretrain on all three sources
- **arm B** - untrained encoder
- **arm C** - from scratch on target labels
- **arm D** - once per individual source, so three runs

That last point is where the run count comes from. Arm D is 180 of the 360
runs, because "does a single source do just as well?" has to be asked of each
source separately - answering it with one arbitrarily chosen source would not
settle anything.

In [2]:
all_configs = arms.matrix_configs()
print(f"total runs in the full protocol: {len(all_configs)}")
print()
display(arms.describe_matrix(all_configs))

# The property that makes it leave-one-domain-out at all.
for cfg in all_configs:
    assert cfg.target_domain not in cfg.source_domains, (
        f"{cfg.run_id}: the target leaked into its own pretraining sources"
    )
print("\nEvery run: the target domain is absent from its pretraining sources.")
print(f"Every run_id unique: {len({c.run_id for c in all_configs}) == len(all_configs)}")

total runs in the full protocol: 360



target,cora,elliptic,photo,ppi
arm,,,,
A_transfer,15,15,15,15
B_random,15,15,15,15
C_scratch,15,15,15,15
D_expert,45,45,45,45



Every run: the target domain is absent from its pretraining sources.
Every run_id unique: True


## 2. One fold, every arm

Target **Cora**, sources Photo / PPI / Elliptic, 100% of labels, seed 0.

Preparation happens once per seed. `src/pipeline.py` caches the loaded graphs
and the structural features -- both split-independent -- but never the fitted
transforms, since the SVD basis and scalers are fit on the training split and
caching those across folds would be precisely the leak notebook 02 tests for.

In [3]:
TARGET = "cora"
SOURCES = pipeline.source_domains_for(TARGET)
BUDGET = arms.Budget.smoke()

print(f"target  : {TARGET}")
print(f"sources : {SOURCES}")
print(f"budget  : {BUDGET.describe()}")
print()

prepared = pipeline.prepare_all(seed=SEED, verbose=False)
for name, p in prepared.items():
    print(f"  {name:<9} X{p.X.shape}  train {len(p.split.train_idx):>6,}  task={p.task}")

target  : cora
sources : ('photo', 'ppi', 'elliptic')
budget  : pretrain 30 steps | probe 300 epochs | scratch 30 epochs



  cora      X(2708, 133)  train  1,624  task=multiclass
  photo     X(7650, 133)  train  4,590  task=multiclass
  ppi       X(56944, 133)  train 34,166  task=multilabel
  elliptic  X(203769, 133)  train 27,938  task=binary


In [4]:
fold_configs = [
    RunConfig(arm="A_transfer", target_domain=TARGET, source_domains=SOURCES,
              seed=SEED, label_fraction=1.0),
    RunConfig(arm="B_random", target_domain=TARGET, source_domains=(),
              seed=SEED, label_fraction=1.0),
    RunConfig(arm="C_scratch", target_domain=TARGET, source_domains=(),
              seed=SEED, label_fraction=1.0),
] + [
    RunConfig(arm="D_expert", target_domain=TARGET, source_domains=(src,),
              seed=SEED, label_fraction=1.0)
    for src in SOURCES
]

cache = {}
outcomes = []
for cfg in fold_configs:
    label = cfg.arm + (f" ({cfg.source_domains[0]})" if cfg.arm == "D_expert" else "")
    print(f"\n{label}")
    outcomes.append(arms.run_arm(cfg, prepared, device, BUDGET,
                                 verbose=True, embedding_cache=cache))


A_transfer
  pretraining on 3 domains round-robin: ['photo', 'ppi', 'elliptic']
    photo      full graph
    ppi        one graph per batch (24 graphs)
    elliptic   2-hop subgraph from 4,096 seeds


    step    1/30  loss 0.9946  disc acc 0.536


    step   25/30  loss 0.4024  disc acc 0.801


    done in 9.0s | best loss 0.3500 at step 28


    -> test accuracy = 0.7841  (9.7s)

B_random
  arm B: untrained encoder, BatchNorm calibrated on cora


...

    step   25/30  loss 0.7083  disc acc 0.513
    early stop at step 29 (no improvement for 10 steps)
    done in 0.8s | best loss 0.6520 at step 19


    -> test accuracy = 0.7934  (1.2s)

D_expert (elliptic)
  pretraining on elliptic | 2-hop subgraph from 4,096 seeds
    step    1/30  loss 1.0018  disc acc 0.439


    step   25/30  loss 0.0421  disc acc 0.997


    done in 5.2s | best loss 0.0160 at step 29


    -> test accuracy = 0.7638  (5.6s)


In [5]:
target_prep = prepared[TARGET]
pm = metrics.primary_metric_for(TARGET, target_prep.task)
baseline = metrics.majority_baseline(
    target_prep.task, target_prep.y[target_prep.split.test_idx]
).get(pm, np.nan)

rows = []
for o in outcomes:
    src = ", ".join(o.config.source_domains) or "-"
    rows.append({
        "arm": o.config.arm,
        "sources": src,
        pm: o.primary_value,
        "vs_baseline": o.primary_value - baseline,
        "converged": o.converged,
        "seconds": round(o.total_seconds, 1),
    })

smoke = pd.DataFrame(rows)
display(smoke)
print(f"\nno-model baseline ({pm}): {baseline:.4f}")

,arm,sources,accuracy,vs_baseline,converged,seconds
0,A_transfer,"photo, ppi, elliptic",0.784133,0.483395,True,9.7
1,B_random,-,0.760148,0.459410,True,0.6
2,C_scratch,-,0.832103,0.531365,False,1.1
3,D_expert,photo,0.760148,0.459410,True,2.9
4,D_expert,ppi,0.793358,0.492620,True,1.2
5,D_expert,elliptic,0.763838,0.463100,True,5.6



no-model baseline (accuracy): 0.3007


### Read this as a smoke test, not a result

Three sanity checks below. They test that nothing is *broken* - not that
anything is good. At 30 pretraining steps and 300 probe epochs, the ordering
between arms carries no information.

In [6]:
assert smoke[pm].notna().all(), "an arm produced NaN"
assert (smoke[pm] > baseline).all(), "an arm failed to beat the no-model baseline"
assert len(smoke) == 6, "expected A, B, C and three D runs"

n_converged = int(smoke["converged"].sum())
print(f"arms that reached a validation plateau: {n_converged} of {len(smoke)}")
print()
if n_converged < len(smoke):
    print("EXPECTED at this budget. A probe capped at 300 epochs on a barely")
    print("pretrained encoder has no reason to plateau. These numbers are")
    print("lower bounds produced by an undersized budget -- they are NOT")
    print("results, and no ranking between arms should be read from them.")
print()
print("What the smoke test actually establishes:")
print("  - all four code paths execute end to end")
print("  - every arm clears the no-model baseline")
print("  - no NaNs, no shape errors, no device errors")
print("  - the embedding cache is being hit across arms")

arms that reached a validation plateau: 5 of 6

EXPECTED at this budget. A probe capped at 300 epochs on a barely
pretrained encoder has no reason to plateau. These numbers are
lower bounds produced by an undersized budget -- they are NOT
results, and no ranking between arms should be read from them.

What the smoke test actually establishes:
  - all four code paths execute end to end
  - every arm clears the no-model baseline
  - no NaNs, no shape errors, no device errors
  - the embedding cache is being hit across arms


## 3. Does BatchNorm recalibration on the target matter?

Arms A and D pretrain on source domains, so their BatchNorm running statistics
describe those sources rather than the target. We recalibrate them on the
target's features before embedding it -- label-free, forward passes only --
because otherwise arm A would be penalised for a *normalisation mismatch*
rather than for anything about its representation, while arm B (calibrated on
the target by construction) would not be.

Notebook 03 showed how large this class of effect can be: an uncalibrated
random encoder on Photo produced embeddings with mean norm 1730 and an
effective rank of 5.3 out of 128. Here is the size of it for transfer.

In [7]:
cfg_a = fold_configs[0]
comparison = []
for calibrate in (False, True):
    out = arms.run_arm(cfg_a, prepared, device, BUDGET,
                       calibrate_target_bn=calibrate, verbose=False)
    comparison.append({
        "target_bn_recalibrated": calibrate,
        pm: out.primary_value,
        "seconds": round(out.total_seconds, 1),
    })

bn_df = pd.DataFrame(comparison)
display(bn_df)
delta = bn_df[pm].iloc[1] - bn_df[pm].iloc[0]
print(f"\neffect of recalibrating BatchNorm on the target: {delta:+.4f}")
print("Reported rather than assumed. It is applied to arms A, B and D alike,")
print("so whichever way it goes, it cannot favour one arm over another.")

,target_bn_recalibrated,accuracy,seconds
0,False,0.787823,8.9
1,True,0.784133,9.1



effect of recalibrating BatchNorm on the target: -0.0037
Reported rather than assumed. It is applied to arms A, B and D alike,
so whichever way it goes, it cannot favour one arm over another.


## 4. How long will the real run take?

Measured from this fold, scaled to the full matrix. The smoke budget is 10x
smaller on pretraining and 10x smaller on probe epochs, so the estimate below
scales the measured times and is deliberately rough - it is a planning number,
not a promise.

In [8]:
per_arm = {}
for o in outcomes:
    per_arm.setdefault(o.config.arm, []).append(o.total_seconds)

# How many runs of each arm the full matrix contains.
counts = {}
for c in all_configs:
    counts[c.arm] = counts.get(c.arm, 0) + 1

# The smoke budget shrinks pretraining 10x and probe epochs 10x. Frozen arms
# also reuse embeddings across the 5 label fractions, so only 1 in 5 pays the
# pretraining cost.
FULL = arms.Budget()
pre_scale = FULL.pretrain_steps / BUDGET.pretrain_steps
probe_scale = FULL.probe_epochs / BUDGET.probe_epochs

est = []
for arm, times in per_arm.items():
    mean_smoke = float(np.mean(times))
    if arm == "C_scratch":
        scale = FULL.scratch_epochs / BUDGET.scratch_epochs
        per_run = mean_smoke * scale
    else:
        # one pretrain per 5 fractions, but a probe every time
        per_run = (mean_smoke * pre_scale) / 5 + mean_smoke * probe_scale
    est.append({
        "arm": arm,
        "smoke_seconds": round(mean_smoke, 1),
        "runs_in_matrix": counts[arm],
        "est_seconds_per_run": round(per_run, 1),
        "est_hours_total": round(per_run * counts[arm] / 3600, 2),
    })

est_df = pd.DataFrame(est).set_index("arm")
display(est_df)
print(f"\nrough total: {est_df['est_hours_total'].sum():.1f} hours on this GPU")
print()
print("Cora is the smallest domain, so this understates the folds targeting")
print("PPI and Elliptic. Treat it as a lower bound on the full run.")

,smoke_seconds,runs_in_matrix,est_seconds_per_run,est_hours_total
arm,,,,
A_transfer,9.7,60,116.3,1.94
B_random,0.6,60,7.2,0.12
C_scratch,1.1,60,11.2,0.19
D_expert,3.2,180,38.9,1.95



rough total: 4.2 hours on this GPU

Cora is the smallest domain, so this understates the folds targeting
PPI and Elliptic. Treat it as a lower bound on the full run.


## 5. Resumability

`results/runs.jsonl` is append-only and every row carries a `run_id`. The
harness skips ids it has already seen, so a crashed sweep restarts where it
stopped instead of redoing finished work.

This is worth testing rather than assuming: a resume that silently re-runs
everything turns a 5-hour job into a 10-hour one, and a resume that skips the
wrong rows corrupts the results table.

We write to a scratch file here so the real `runs.jsonl` stays untouched.

In [9]:
import tempfile

scratch = pathlib.Path(tempfile.mkdtemp()) / "runs_smoke.jsonl"

for cfg, out in zip(fold_configs, outcomes):
    config.log_run(cfg, out.as_metrics_dict(), path=scratch)

logged = config.load_runs(scratch)
done = config.completed_run_ids(scratch)

print(f"rows written : {len(logged)}")
print(f"unique ids   : {len(done)}")
print()

remaining = [c for c in all_configs if c.run_id not in done]
print(f"full matrix          : {len(all_configs)}")
print(f"already completed    : {len(all_configs) - len(remaining)}")
print(f"a resume would run   : {len(remaining)}")

assert len(all_configs) - len(remaining) == len(fold_configs), (
    "resume logic did not recognise exactly the runs we logged"
)
print("\nResume skips exactly the finished runs, and no others.")

row = logged[0]
print(f"\nexample row keys: {sorted(row)}")
print(f"  run_id      : {row['run_id']}")
print(f"  config_hash : {row['config_hash']}")
print(f"  git_commit  : {row['git_commit']}")
print(f"  primary     : {row['metrics']['primary_metric']} = "
      f"{row['metrics']['primary_value']:.4f}")

rows written : 6
unique ids   : 6

full matrix          : 360
already completed    : 6
a resume would run   : 354

Resume skips exactly the finished runs, and no others.

example row keys: ['config', 'config_hash', 'git_commit', 'metrics', 'platform', 'run_id', 'seed', 'timestamp']
  run_id      : A_transfer__cora__lf1__s0__5a40cf00a453
  config_hash : 5a40cf00a453
  git_commit  : 7a5ddcc
  primary     : accuracy = 0.7841


## 6. Reproducibility, checked here rather than hoped for

The claim that a config hash plus a seed re-derives a row is the foundation of
the whole results table. It is also easy to lose by accident, and it *was*
lost twice during development - once because `run_arm` did not reseed, once
because CUDA scatter-add is non-deterministic.

Below, the same config is run twice with another arm interleaved between them
specifically to disturb the global RNG state.

In [10]:
probe_cfg = fold_configs[1]        # arm B: cheap to repeat
disturber = fold_configs[3]        # arm D: perturbs the RNG between repeats

first = arms.run_arm(probe_cfg, prepared, device, BUDGET, verbose=False)
arms.run_arm(disturber, prepared, device, BUDGET, verbose=False)
second = arms.run_arm(probe_cfg, prepared, device, BUDGET, verbose=False)

print(f"run 1: {first.primary_value:.10f}")
print(f"run 2: {second.primary_value:.10f}   (with another arm run in between)")
assert first.primary_value == second.primary_value, "runs are not reproducible"
print("\nIdentical. A row of runs.jsonl can be re-derived from its config and seed.")

run 1: 0.7601476015
run 2: 0.7601476015   (with another arm run in between)

Identical. A row of runs.jsonl can be re-derived from its config and seed.


## What you should now understand

- **The full protocol is 360 runs, and arm D is half of them.** Arm D asks
  "would a single source have done just as well?", and that question has to be
  put to each source separately - which is why it costs 180 runs rather than
  60. Dropping it would make "arm A beats arm B" mean only that *some*
  pretraining helps.
- **A smoke test checks that code runs, not that results are good.** Every arm
  executes, clears the no-model baseline and produces no NaNs. At 30
  pretraining steps nothing has converged, so no ranking between arms here
  means anything, and the convergence flags say so.
- **Reproducibility and resumability are properties you test, not properties
  you assume.** Both were broken at some point in development and both are now
  asserted mechanically: the same config and seed give bit-identical numbers,
  and a resume skips exactly the finished runs.

**Next:** `scripts/run_all.py` executes the full matrix headless and
resumable, then `06_results.ipynb` reads `results/runs.jsonl` and builds the
tables and figures.